# Fruit & Vegetable Classification using Deep Learning (CNN)

In [49]:
from google.colab import drive

drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [27]:
from PIL import Image
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.models import Sequential

In [28]:
img_height = 180
img_width = 180
batch_size = 32

In [29]:
train_path = "/content/drive/MyDrive/Fruits_Vegetables/train"
validation_path = "/content/drive/MyDrive/Fruits_Vegetables/validation"

In [30]:
# Load the Training Dataset

train_images = tf.keras.utils.image_dataset_from_directory(
    train_path,
    image_size=(img_height, img_width),
    batch_size=batch_size,
    shuffle=True,
    seed=42
)

Found 2341 files belonging to 36 classes.


In [31]:
# laod validation dataset
validation_images = tf.keras.utils.image_dataset_from_directory(
    validation_path,
    image_size=(img_height, img_width),
    batch_size=batch_size,
    shuffle=False
)

Found 347 files belonging to 36 classes.


In [33]:
# labels
data_labels = train_images.class_names

In [34]:
print("Classes:", data_labels)
print("Total classes:", len(data_labels))

Classes: ['apple', 'banana', 'beetroot', 'bell pepper', 'cabbage', 'capsicum', 'carrot', 'cauliflower', 'chilli pepper', 'corn', 'cucumber', 'eggplant', 'garlic', 'ginger', 'grapes', 'jalepeno', 'kiwi', 'lemon', 'lettuce', 'mango', 'onion', 'orange', 'paprika', 'pear', 'peas', 'pineapple', 'pomegranate', 'potato', 'raddish', 'soy beans', 'spinach', 'sweetcorn', 'sweetpotato', 'tomato', 'turnip', 'watermelon']
Total classes: 36


In [24]:
# CNN Model
model = Sequential([
    layers.Input(shape=(img_height, img_width, 3)),
    layers.Rescaling(1./255),

    layers.Conv2D(16, 3, padding="same", activation="relu"),
    layers.MaxPooling2D(),

    layers.Conv2D(32, 3, padding="same", activation="relu"),
    layers.MaxPooling2D(),

    layers.Conv2D(64, 3, padding="same", activation="relu"),
    layers.MaxPooling2D(),

    layers.Dropout(0.2),
    layers.Flatten(),
    layers.Dense(128, activation="relu"),
    layers.Dense(len(data_labels))
])

In [35]:
# Compile the Model
model.compile(
    optimizer="adam",
    loss=tf.keras.losses.SparseCategoricalCrossentropy(
        from_logits=True
    ),
    metrics=["accuracy"]
)


In [36]:
# Training the Model

history = model.fit(
    train_images,
    validation_data=validation_images,
    epochs=3
)

Epoch 1/3
74/74 ━━━━━━━━━━━━━━━━━━━━ 137s 2s/step - accuracy: 0.2046 - loss: 2.7584 - val_accuracy: 0.3602 - val_loss: 4.1674
Epoch 2/3
74/74 ━━━━━━━━━━━━━━━━━━━━ 41s 558ms/step - accuracy: 0.3106 - loss: 2.3751 - val_accuracy: 0.4207 - val_loss: 4.3974
Epoch 3/3
74/74 ━━━━━━━━━━━━━━━━━━━━ 74s 456ms/step - accuracy: 0.3870 - loss: 2.0762 - val_accuracy: 0.4726 - val_loss: 4.5428


In [55]:

# Prediction

import numpy as np

def predictor(img):
    img = tf.keras.utils.load_img(img, target_size=(img_height, img_width))

    img_array = tf.keras.utils.img_to_array(img)
    img_array = tf.expand_dims(img_array, 0)
    pred = model.predict(img_array, verbose=0)
    score = tf.nn.softmax(pred[0])

    return (f"""The provided image is a {data_labels[np.argmax(score)]} and model confidence is {np.max(score)*100}""")

In [56]:
# Predict the img

predictor("/content/drive/MyDrive/Fruits_Vegetables/validation/banana/Image_1.jpg")

'The provided image is a banana and model confidence is 21.4111270904541'

In [43]:
data_labels

['apple',
 'banana',
 'beetroot',
 'bell pepper',
 'cabbage',
 'capsicum',
 'carrot',
 'cauliflower',
 'chilli pepper',
 'corn',
 'cucumber',
 'eggplant',
 'garlic',
 'ginger',
 'grapes',
 'jalepeno',
 'kiwi',
 'lemon',
 'lettuce',
 'mango',
 'onion',
 'orange',
 'paprika',
 'pear',
 'peas',
 'pineapple',
 'pomegranate',
 'potato',
 'raddish',
 'soy beans',
 'spinach',
 'sweetcorn',
 'sweetpotato',
 'tomato',
 'turnip',
 'watermelon']

In [58]:
# Saved & Export the Model
model.save("Fruit_Veg_Model.keras")